# MAXIMUM ENTROPY MODEL

In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

In [2]:
df = pd.read_csv('../DATASETS/SA_preprocessed.csv')
df = df.dropna(subset=['processed_text']).reset_index(drop=True)

In [3]:
print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")

Dataset shape: (6043, 4)
Label distribution:
label
0    4149
1    1894
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df['processed_text'].values,
    df['label'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['label'].values # splits the dataset such that the spam to ham ratio is similar in training and test sets
)

X_train[0]

're information that you have requested number e earn moneylimit number or more per week this offer is limited to the first number people who contact me today let us face it every business opportunity is not for everyone you need something that fits your needs budget and schedule that is why we have put together several real income opportunities just for you we have searched and searched and finally found and compiled the best opportunities available i promise you will not regret it you will finally find something you truly can make money with you really can make an extra moneylimit number to moneylimit number number a week if you have a few hours a week to work your business you do not have to pay one dime to find out about these true money making opportunities just call number number number number and we will show you the best real moneymakers available it is number free so visit us today do not miss out on a life changing opportunity this is absolutely no risk so call number number 

In [5]:
print(f"Training set: {len(X_train)}")
print(f"Test set: {len(X_test)}")

Training set: 4834
Test set: 1209


In [ ]:
rules = [
    # Core spam words
    ("contains_free",        lambda t: 'free' in t),
    ("contains_money",       lambda t: any(w in t for w in ['money', 'cash'])),
    ("contains_offer",       lambda t: any(w in t for w in ['offer', 'deal'])),
    ("contains_urgent",      lambda t: any(w in t for w in ['urgent', 'act now', 'hurry'])),
    ("contains_credit",      lambda t: any(w in t for w in ['credit', 'loan', 'mortgage'])),
    ("contains_email",       lambda t: 'emailaddr' in t),
    ("contains_guarantee",   lambda t: any(w in t for w in ['guarantee', 'guaranteed'])),
    ("contains_limited_time",lambda t: 'limited time' in t),
    ("contains_click_here",  lambda t: any(w in t for w in ['click here', 'click below'])),
    ("contains_remove",      lambda t: any(w in t for w in ['remove', 'unsubscribe'])),
    ("contains_work_home",   lambda t: any(w in t for w in ['work from home', 'home based', 'earn from home'])),
    ("contains_mlm",         lambda t: any(w in t for w in ['mlm', 'multi-level'])),
    ("contains_weight",      lambda t: any(w in t for w in ['weight', 'diet', 'lose fat'])),
    ("contains_bizop",       lambda t: any(w in t for w in ['bizop', 'business opportunity'])),
    ("dear_sir_madam",       lambda t: 'dear' in t and any(w in t for w in ['sir', 'madam'])),
    ("click_watch",          lambda t: 'click' in t and any(w in t for w in ['watch', 'see', 'view'])),
    ("here_url",             lambda t: 'here' in t and any(w in t for w in ['http', 'www', 'urladdr'])),
    # ("contains_hyperlink",             lambda t: 'hyperlink' in t),
    # ("contains_number",             lambda t: 'hyperlink' in t),
    # ("multiple_hyperlinks",        lambda t: t.count('hyperlink') >= 2),
    # ("multiple_number",        lambda t: t.count('number') >= 2),

    # may not be spam
    ("contains_ilug",        lambda t: 'ilug' in t),

    # Structural / contextual
    ("multiple_urls",        lambda t: t.count('http') >= 2 or t.count('urladdr') >= 2),
    ("reply_to",             lambda t: 'reply' in t and 'to' in t),
    ("sent_you",             lambda t: 'sent' in t and 'you' in t),
    ("contains_university",  lambda t: any(w in t for w in ['university', 'edu', 'professor'])),
    ("question_help",        lambda t: any(w in t for w in ['question', 'help', 'problem'])),
    ("has_webcam",           lambda t: 'webcam' in t),
    ("hello_name",           lambda t: 'hello' in t and 'name' in t),

    ("very_short",           lambda t: len(t) < 200),
    ("sent_from",            lambda t: 'sent' in t and 'from' in t),
    ("original_message",     lambda t: 'original' in t and 'message' in t),
    ("on_wrote",             lambda t: 'on' in t and 'wrote' in t),
    ("make_money",           lambda t: 'make' in t and 'money' in t),
    ("has_opportunity",      lambda t: 'opportunity' in t),
]

'''Why rules shoudn't be merged: Each rule gets its own independent weight during training — that's the whole point of MaxEnt. For example:

If you collapse them into one rule like "has_spam_words", the model can only assign a single weight to the whole group, 
losing the ability to distinguish strong signals from weak ones. Accuracy would drop.

They look repetitive in code, but they're separate features to the model — each one matters independently.'''


rule_names = [name for name, _ in rules]
rule_funcs = [fn for _, fn in rules]
n_rules = len(rules)
print(f"\nTotal rules: {n_rules}")


Total rules: 33


In [18]:
# Feature extraction
def extract_features(text):
    """Extract binary feature vector from a single text."""
    return np.array([1.0 if fn(text) else 0.0 for fn in rule_funcs])

def extract_features_batch(texts):
    """Extract feature matrix for a list of texts."""
    X = np.zeros((len(texts), n_rules))
    for i, text in enumerate(texts):
        X[i] = extract_features(text)
    return X

print(extract_features(X_train[0]))
print(len(extract_features(X_train[0])))

[1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1. 0. 0. 0. 0. 1.
 0. 0. 0. 0. 0. 0. 0. 1. 1.]
33


In [19]:
def predict_proba(X_features, weights):
    # P(spam|x) = sigmoid(w·x + b)
    logits = X_features @ weights[1:] + weights[0]
    return 1.0 / (1.0 + np.exp(-logits))

def predict(X_features, weights, threshold=0.5):
    # Return 0/1 labels.
    return (predict_proba(X_features, weights) >= threshold).astype(int)

def train(X_features, y, lr=0.1, epochs=5000, l2_lambda=0.001):
    # Train using Stochastic Gradient Descent. Returns weights array.
    n_samples, n_features = X_features.shape
    weights = np.zeros(n_features + 1)  # weights[0] = bias

    for epoch in range(epochs):
        # forward pass
        logits = X_features @ weights[1:] + weights[0]
        preds = 1.0 / (1.0 + np.exp(-logits))
        errors = preds - y

        # gradients
        grad_bias = np.mean(errors)
        grad_weights = X_features.T @ errors / n_samples + 2 * l2_lambda * weights[1:]

        # update
        weights[0] -= lr * grad_bias
        weights[1:] -= lr * grad_weights

        # print loss every 200 epochs
        if epoch % 200 == 0:
            loss = np.mean(np.log(1.0 + np.exp(-(2 * y - 1) * logits)))
            loss += l2_lambda * np.sum(weights[1:] ** 2)
            print(f"  Epoch {epoch:4d} | Loss: {loss:.4f}")

    return weights

In [20]:
print("\nExtracting features...")

X_train_feat = extract_features_batch(X_train)
X_test_feat = extract_features_batch(X_test)

print("\nTraining Maximum Entropy model...")

weights = train(X_train_feat, y_train)

print(f"\nBias (intercept): {weights[0]:.4f}")


Extracting features...

Training Maximum Entropy model...
  Epoch    0 | Loss: 0.6931
  Epoch  200 | Loss: 0.3530
  Epoch  400 | Loss: 0.3210
  Epoch  600 | Loss: 0.3091
  Epoch  800 | Loss: 0.3025
  Epoch 1000 | Loss: 0.2982
  Epoch 1200 | Loss: 0.2953
  Epoch 1400 | Loss: 0.2931
  Epoch 1600 | Loss: 0.2915
  Epoch 1800 | Loss: 0.2903
  Epoch 2000 | Loss: 0.2894
  Epoch 2200 | Loss: 0.2887
  Epoch 2400 | Loss: 0.2882
  Epoch 2600 | Loss: 0.2878
  Epoch 2800 | Loss: 0.2874
  Epoch 3000 | Loss: 0.2871
  Epoch 3200 | Loss: 0.2869
  Epoch 3400 | Loss: 0.2868
  Epoch 3600 | Loss: 0.2866
  Epoch 3800 | Loss: 0.2865
  Epoch 4000 | Loss: 0.2864
  Epoch 4200 | Loss: 0.2863
  Epoch 4400 | Loss: 0.2863
  Epoch 4600 | Loss: 0.2862
  Epoch 4800 | Loss: 0.2862

Bias (intercept): -1.5057


In [21]:
pairs = [(rule_names[i], weights[i + 1]) for i in range(n_rules)]
pairs.sort(key=lambda x: abs(x[1]), reverse=True)

print("\nSPAM indicators (positive weight)")
for name, w in pairs:
    if w > 0:
        print(f"  {name:30s}  {w:+.4f}")

print("\nHAM indicators (negative weight)")
for name, w in pairs:
    if w <= 0:
        print(f"  {name:30s}  {w:+.4f}")


SPAM indicators (positive weight)
  contains_click_here             +1.8135
  reply_to                        +1.6225
  contains_guarantee              +1.4401
  dear_sir_madam                  +1.0988
  contains_credit                 +1.0868
  contains_urgent                 +1.0595
  has_opportunity                 +1.0060
  contains_hyperlink              +0.9174
  contains_number                 +0.9174
  contains_money                  +0.8755
  contains_remove                 +0.8472
  contains_work_home              +0.6456
  sent_you                        +0.5872
  contains_offer                  +0.5822
  contains_free                   +0.5468
  contains_weight                 +0.4761
  contains_limited_time           +0.3288
  contains_mlm                    +0.3112
  hello_name                      +0.2942
  contains_bizop                  +0.2040

HAM indicators (negative weight)
  on_wrote                        -2.1822
  original_message                -0.7490
  conta

In [22]:
# Evaluate
y_pred = predict(X_test_feat, weights)
accuracy = accuracy_score(y_test, y_pred)

print(f"  RESULTS")
print(f"Accuracy: {accuracy:.4f}  ({accuracy:.2%})\n")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))
print(f"F1 score:{f1_score(y_test, y_pred)}")

cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:")
print(f"                 Predicted")
print(f"              Ham    Spam")
print(f"Actual Ham   {cm[0,0]:4d}   {cm[0,1]:4d}")
print(f"       Spam  {cm[1,0]:4d}   {cm[1,1]:4d}")

# Misclassified emails
misclassified = np.where(y_pred != y_test)[0]
probas = predict_proba(X_test_feat, weights)

print(f"\n\nMisclassified: {len(misclassified)} / {len(y_test)}")

for idx in misclassified[:10]:
    actual = "SPAM" if y_test[idx] == 1 else "HAM"
    predicted = "SPAM" if y_pred[idx] == 1 else "HAM"
    print(f"  Actual: {actual:4s} | Predicted: {predicted:4s} | P(spam): {probas[idx]:.2%}")
    print(f"  Text: {X_test[idx]}...")
    print()

  RESULTS
Accuracy: 0.9074  (90.74%)

              precision    recall  f1-score   support

         Ham       0.92      0.95      0.93       830
        Spam       0.89      0.81      0.85       379

    accuracy                           0.91      1209
   macro avg       0.90      0.88      0.89      1209
weighted avg       0.91      0.91      0.91      1209

F1 score:0.8453038674033149
Confusion Matrix:
                 Predicted
              Ham    Spam
Actual Ham    791     39
       Spam    73    306


Misclassified: 112 / 1209
  Actual: SPAM | Predicted: HAM  | P(spam): 39.37%
  Text: adv put your resume back to work dear candidate we recently came across a posting of your resume on the internet after our research we determined that it was not posted on over number other top job sites the impact to you is that over number of the employers and recruiters looking to hire candidates with your skills are not reading your resume is your resume taking the day off while most people p

In [28]:
# user_input = input("Enter email text: ")


#processed = "re nlp evaluation i love nlp this is not spam work for money free money from vivesh hyperlink number number hyperlink"
processed = "re nlp evaluation i love nlp from vivesh hyperlink number number hyperlink"
features = np.array([1.0 if fn(processed) else 0.0 for fn in rule_funcs])
prob = 1.0 / (1.0 + np.exp(-(features @ weights[1:] + weights[0])))
prediction = 'Spam' if prob > 0.5 else 'Ham'
print(f"Probability: {prob:.4f}, Prediction: {prediction}")

Probability: 0.4711, Prediction: Ham
